In [11]:

import json
from openai import OpenAI, RateLimitError, APIConnectionError, APIStatusError
from google.colab import userdata

# ==========================================
# 1. API CONFIGURATION
# ==========================================

try:
    api_key = userdata.get("Groq_key")
except Exception:
    api_key = None

if not api_key:
    raise ValueError(
        "Groq API key not found. Add Groq_key in Colab Secrets."
    )

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-20b"


# ==========================================
# 2. BAKERY MENU
# ==========================================

MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}


# ==========================================
# 3. SYSTEM PROMPT
# ==========================================

SYSTEM = f"""
ROLE:
You are a friendly AI assistant for Sweet Crumbs Home Bakery.

FACTS:
Menu and prices in rupees:
{json.dumps(MENU)}

- Orders require 24 hours' notice.
- All cakes have eggless versions.

RULES:
1. Never promise a delivery time.
2. If information is unavailable, say:
   "I'll check with the owner."
3. Remember the customer's name and order during the chat.
4. Use the get_price tool whenever a customer
   asks for a product price or order total.
5. Never invent menu items or prices.
6. Do not claim that an order is placed or confirmed.
   You can only assist with enquiries.
7. Match the customer's language (English or Telugu).

FORMAT:
Keep responses friendly and under 50 words.
"""


# ==========================================
# 4. PRICE CALCULATION FUNCTION
# ==========================================

def get_price(item: str, qty: int = 1) -> str:

    item = item.lower().strip()

    if item not in MENU:
        return f"Sorry, {item} is not available on our menu."

    if not isinstance(qty, int) or isinstance(qty, bool) or qty <= 0:
        return "Please provide a valid positive quantity."

    total = MENU[item] * qty

    print(f"[LOG] get_price({item}, {qty}) = ₹{total}")

    return json.dumps({
        "item": item,
        "quantity": qty,
        "unit_price": MENU[item],
        "total_price": total
    })


# ==========================================
# 5. DEFINE TOOL SCHEMA
# ==========================================

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_price",
            "description": "Calculate the total price of bakery products.",
            "parameters": {
                "type": "object",
                "properties": {
                    "item": {
                        "type": "string",
                        "enum": list(MENU.keys()),
                        "description": "Name of the bakery product."
                    },
                    "qty": {
                        "type": "integer",
                        "minimum": 1,
                        "description": "Number of items."
                    }
                },
                "required": ["item", "qty"],
                "additionalProperties": False
            }
        }
    }
]


# ==========================================
# 6. CONVERSATION HISTORY
# ==========================================

history = [
    {
        "role": "system",
        "content": SYSTEM
    }
]

MAX_TURNS = 8


# ==========================================
# 7. TOOL EXECUTION
# ==========================================

def execute_tool(tool_call):

    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    if function_name == "get_price":
        return get_price(
            item=arguments["item"],
            qty=arguments["qty"]
        )

    return "Unknown function."


# ==========================================
# 8. CHAT FUNCTION WITH TOOL CALLING
# ==========================================

def chat(user_text: str) -> str:

    history.append({
        "role": "user",
        "content": user_text
    })

    try:

        for _ in range(5):

            response = client.chat.completions.create(
                model=MODEL,
                messages=history,
                tools=tools,
                tool_choice="auto",
                temperature=0.3
            )

            message = response.choices[0].message

            # Save assistant response
            history.append(
                message.model_dump(exclude_none=True)
            )

            # Check whether AI requested a tool
            if not message.tool_calls:

                reply = message.content or "How can I help you?"

                # Keep the latest 8 complete user turns
                turns = []
                for msg in history[1:]:
                    if msg["role"] == "user":
                        turns.append([msg])
                    elif turns:
                        turns[-1].append(msg)

                history[:] = (
                    [history[0]]
                    + [msg for turn in turns[-MAX_TURNS:] for msg in turn]
                )

                return reply

            # Execute requested tools
            for tool_call in message.tool_calls:

                result = execute_tool(tool_call)

                history.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(result)
                })

        return "I couldn't complete that request. Please try again."

    except RateLimitError:
        history.pop()
        return "We're a little busy. Please try again in a minute."

    except APIConnectionError:
        history.pop()
        return "I can't reach the server. Please try again shortly."

    except APIStatusError as e:
        history.pop()
        print("[API ERROR]", e)
        return "Sorry, the AI service encountered an error."

    except (ValueError, KeyError, TypeError, json.JSONDecodeError) as e:
        print("[TOOL ERROR]", e)
        return "Sorry, I couldn't process that request."


# ==========================================
# 9. TEST PRICE FUNCTION
# ==========================================

print("Testing price function:")

print("Total for 6 cupcakes:")
print(get_price("cupcake", 6))


# ==========================================
# 10. CHATBOT LOOP
# ==========================================

print("\n🍰 Welcome to Sweet Crumbs Home Bakery!")
print("Type 'bye' to exit.\n")

while True:

    user_input = input("You: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ("bye", "exit", "quit"):

        print("Bot: Thank you for visiting Sweet Crumbs!")
        break

    reply = chat(user_input)

    print("Bot:", reply)


Testing price function:
Total for 6 cupcakes:
[LOG] get_price(cupcake, 6) = ₹480
{"item": "cupcake", "quantity": 6, "unit_price": 80, "total_price": 480}

🍰 Welcome to Sweet Crumbs Home Bakery!
Type 'bye' to exit.

You: hi
Bot: Hello! 👋 How can I help you with Sweet Crumbs today?
You: exit
Bot: Thank you for visiting Sweet Crumbs!
